In [1]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[05/19/26 07:50:23] INFO     Using                                                                  ]8;id=429003;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=412414;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\framework\                
                             project\rich_logging.yml' as logging configuration.                                   

[05/19/26 07:50:26] WARNING  c:\Users\User\miniconda3\envs\unis\Lib\site-packages\requests\__init__ ]8;id=902619;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=82884;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             .py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                        
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[05/19/26 07:50:30] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=301563;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=725916;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [2]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import unis_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import unis_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

In [3]:
import unis_perm_flow.pipelines.data_processing.nodes as nodes_dproc

# Filtrar para ver solo funciones y omitir clases o imports
functions_list = [f for f in dir(nodes_dproc) if callable(getattr(nodes_dproc, f)) and not f.startswith("__")]

print("Funciones encontradas en nodes_dproc:")
for func in functions_list:
    print(f"- {func}")

Funciones encontradas en nodes_dproc:
- Tuple
- check_and_export_duplicates
- check_dataframe
- clean_column_names
- clean_column_objects
- clean_nulls
- convert_all_standardized_dates
- convert_standardized_dates
- datetime
- numeric_conversion_node
- remove_accents
- remove_accents_and_special_chars
- select_columns
- unis_preprocessing_calaca
- unis_preprocessing_estaca


# Calendario

In [4]:
calendario = catalog.load("unis_calendario_extendido_uptoday")

[05/19/26 07:50:31] INFO     Loading data from unis_calendario_extendido_uptoday               ]8;id=728207;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=728308;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

# Académico

In [5]:
academico_path = 'data/01_raw/Inscripcion_Oracle.xlsx'
academico_df = pd.read_excel(academico_path)
academico_df = nodes_dproc.clean_column_names(academico_df)
academico_df = nodes_dproc.clean_column_objects(academico_df,'correo_e')
academico_df.rename(columns={
                             'nombre':'nombre_nuevo',
                             'doc_id':'documento',
                             'descr': 'programa_completo',
                             'telefono':'telefono',
                             'estado': 'estado',
                             'ccl_admis': 'periodo_ingreso'}, inplace=True)
academico_df = academico_df.loc[:,['id',
                                    'nombre_nuevo',
                                    'tipo_doc_id',
                                    'documento',
                                    'programa_completo',
                                    'telefono',
                                    'estado',
                                    'periodo_ingreso']]

# Inscritos Unis

In [6]:
inscritos_path = 'data/01_raw/Inscripciones_Materias.xlsx'
inscritos_df = pd.read_excel(inscritos_path)
inscritos_df = nodes_dproc.clean_column_names(inscritos_df)
inscritos_df = nodes_dproc.clean_column_objects(inscritos_df,'correo')

In [7]:
inscritos_df = inscritos_df.loc[:,['documento', 
                                   'nombre',
                                   'correo',
                                   'programa',
                                   'periodo_ingreso',
                                   'ciclo',
                                   'estado_ciclo',
                                   'fase_actual']]
inscritos_df = inscritos_df.sort_values(by=['documento', 'periodo_ingreso', 'ciclo'], ascending=[True, True, True])

In [8]:
inscritos_df_inicios = inscritos_df.drop_duplicates(subset=['documento', 'periodo_ingreso'], keep='first')

# Cruce Acádemico e inscritos

In [9]:
# ===========================================================================
# MERGE OUTER — Conserva el 100% de filas de ambas bases.
# Izquierda: inscritos_df_inicios → define el esquema base de salida.
# right_only → filas de académico sin inscrito (nuevos a analizar).
# left_only  → inscritos sin ficha académica (gaps a investigar).
# ===========================================================================

df_mes_0 = pd.merge(
    inscritos_df_inicios,          # LEFT  → esquema y filas base
    academico_df[[                  # RIGHT → solo columnas necesarias
        'documento',
        'periodo_ingreso',
        'nombre_nuevo',
        'tipo_doc_id',
        'programa_completo',
        'telefono',
        'estado',
        'id'
    ]],
    on=['documento', 'periodo_ingreso'],
    how='outer',                    # Conserva filas de AMBAS bases
    indicator=True
)

# ===========================================================================
# ORDEN DE COLUMNAS: esquema de inscritos primero, enriquecimiento después.
# Las filas right_only tendrán NaN en columnas de inscritos — esperado.
# ===========================================================================

df_mes_0 = df_mes_0[[
    'documento',
    'nombre',
    'correo',
    'programa',
    'periodo_ingreso',
    'ciclo',
    'estado_ciclo',
    'fase_actual',
    'nombre_nuevo',
    'tipo_doc_id',
    'programa_completo',
    'telefono',
    'estado',
    'id',
    '_merge'
]]

# ===========================================================================
# QC: distribución del merge — las tres categorías deben tener sentido
# de negocio antes de continuar el pipeline.
# ===========================================================================

cobertura = df_mes_0['_merge'].value_counts()

print(f"[QC] Total registros          : {len(df_mes_0):,}")
print(f"[QC] Match completo (both)    : {cobertura.get('both', 0):,}")
print(f"[QC] Solo en inscritos        : {cobertura.get('left_only', 0):,}")
print(f"[QC] Solo en académico        : {cobertura.get('right_only', 0):,}")

# ===========================================================================
# ALERTA DE DUPLICADOS: el outer puede inflar filas si academico_df
# tiene múltiples registros para el mismo (documento, periodo_ingreso).
# ===========================================================================

dupes = df_mes_0.duplicated(subset=['documento', 'periodo_ingreso','programa']).sum()
if dupes > 0:
    print(f"[ALERTA] {dupes} duplicados — cardinalidad N:1 en academico_df, revisar antes de continuar")
else:
    print("[QC] Sin duplicados en clave compuesta ✓")

[QC] Total registros          : 1,816
[QC] Match completo (both)    : 1,703
[QC] Solo en inscritos        : 0
[QC] Solo en académico        : 113
[ALERTA] 4 duplicados — cardinalidad N:1 en academico_df, revisar antes de continuar


In [11]:
113/1816

0.06222466960352423

In [10]:
# ===========================================================================
# keep=False → muestra TODAS las filas del grupo duplicado, no solo las
# repetidas. Con keep='first' (default) el registro original no aparece
# y el diagnóstico queda incompleto.
# ===========================================================================

mask_dupes = academico_df.duplicated(subset=['documento', 'periodo_ingreso'], keep=False)

duplicados = (
    academico_df[mask_dupes]
    .sort_values(['documento', 'periodo_ingreso'])  # Agrupa visualmente el par
)

print(f"[QC] Filas en grupos duplicados: {len(duplicados):,}")
print(f"[QC] Claves únicas duplicadas  : {duplicados.groupby(['documento', 'periodo_ingreso']).ngroups:,}")

duplicados

[QC] Filas en grupos duplicados: 8
[QC] Claves únicas duplicadas  : 4


,id,nombre_nuevo,tipo_doc_id,documento,programa_completo,telefono,estado,periodo_ingreso
569,22917,"xiloj perez,luis alfonso",dpi,1733051801413,ma neuropsicologia clinica,57679502,dc,9251
572,22917,"xiloj perez,luis alfonso",dpi,1733051801413,ma en docencia universitaria 2,57679502,dc,9251
570,22917,"xiloj perez,luis alfonso",dpi,1733051801413,ma neuropsicologia clinica 2,57679502,ac,9265
571,22917,"xiloj perez,luis alfonso",dpi,1733051801413,ma en docencia universitaria,57679502,ac,9265
787,23663,"coburger ortega,haudrey ann",dpi,2398626000101,ma marketing digital,52080558,ac,9252
788,23663,"coburger ortega,haudrey ann",dpi,2398626000101,ma en docencia universitaria 2,52080558,ac,9252
1124,25501,"reyna franco,astrid",dpi,2664824730101,ma marketing digital,57197722,ac,9253
1125,25501,"reyna franco,astrid",dpi,2664824730101,ma en docencia universitaria 2,57197722,ac,9253


# Base longitudinal

In [48]:
# =============================================================================
# ETL: Base Longitudinal — Todos los estudiantes × Calendario UNIS
#
# Lógica de cruce:
#   inscritos_df.periodo_ingreso  →  calendario.periodo_inicial  (cohorte del estudiante)
#   inscritos_df.ciclo            →  calendario.periodo_actual   (ciclo cursado)
#
# Resultado: una fila por (estudiante × semana académica cursada)
# =============================================================================

base_longitudinal = (
    inscritos_df
    # ── 1. Join por cohorte: ancla el estudiante a su calendario base ─────
    .merge(
        calendario,
        left_on="periodo_ingreso",
        right_on="periodo_inicial",
        how="inner"           # solo cohortes con calendario definido
    )
    # ── 2. Filtrar: mantener solo las semanas del ciclo que cursó ─────────
    # Sin este filtro cada estudiante tendría TODAS las semanas de su cohorte,
    # incluyendo ciclos que aún no cursó o no le corresponden.
    .query("ciclo == periodo_actual")
    # ── 3. Ordenar por estudiante y tiempo ────────────────────────────────
    .sort_values(["documento", "semana_acumulada"])
    .reset_index(drop=True)
)

# ── 4. journey_step: contador secuencial por estudiante ───────────────────
# Diferente a semana_acumulada: es el ordinal propio del estudiante (1, 2, 3...)
# independiente de recesos o saltos en semana_acumulada entre ciclos.
base_longitudinal["journey_step"] = (
    base_longitudinal
    .groupby("documento")
    .cumcount() + 1
)

# ── 5. Seleccionar y ordenar columnas finales ─────────────────────────────
cols_identidad  = ["documento", "nombre", "correo", "programa"]
cols_inscripcion = ["periodo_ingreso", "ciclo", "estado_ciclo", "fase_actual"]
cols_calendario = [
    "periodo_raw", "cohorte", "periodo_inicial", "periodo_actual",
    "sesion", "tipo", "student_journey",
    "fecha_ingreso", "fecha_inicio", "fecha_fin", "shifted_fecha_inicio",
    "semana", "semana_acumulada", "month", "mes_academico",
    "anio_gregoriano", "mes_gregoriano",
]
cols_tiempo     = ["journey_step"]

base_longitudinal = base_longitudinal[
    cols_identidad + cols_inscripcion + cols_calendario + cols_tiempo
]

# ── 6. QC rápido ──────────────────────────────────────────────────────────
print("Shape:", base_longitudinal.shape)
print(f"Estudiantes únicos:       {base_longitudinal['documento'].nunique()}")
print(f"Ciclos únicos cubiertos:  {sorted(base_longitudinal['periodo_actual'].unique())}")
print(f"Rango semana_acumulada:   {base_longitudinal['semana_acumulada'].min()} → {base_longitudinal['semana_acumulada'].max()}")
print(f"Rango journey_step:       {base_longitudinal['journey_step'].min()} → {base_longitudinal['journey_step'].max()}")
print(f"\nObs. por estudiante:")
print(base_longitudinal.groupby("documento")["journey_step"].max().to_string())

Shape: (44813, 26)
Estudiantes únicos:       1691
Ciclos únicos cubiertos:  [np.int64(9243), np.int64(9251), np.int64(9252), np.int64(9253), np.int64(9254), np.int64(9261), np.int64(9262), np.int64(9265)]
Rango semana_acumulada:   1 → 64
Rango journey_step:       1 → 65

Obs. por estudiante:
documento
01050012550      16
0108200414389    17
0316197300120    64
054921258        16
0887628           9
1573659130101    17
1575032750901     1
1575870960101    33
1575874280101    49
1576391420101    64
1576420890101    16
1576545830101    17
1577609440901    16
1578610230101     9
1579733012001    16
1579913340105    32
1580825751106    64
1582596880406    16
1582890800101    64
1584057570101    16
1584812750507    16
1584980460101    16
1585058670101    64
1585791350101    16
1587259051216    16
1587428180101    16
1587669382201     1
1587669620701     1
1588321750101    16
1588635691708     9
1588645062010    16
1588771360101    16
1589577090101    17
1590567640101     9
1591049630901    

In [24]:
mask_documentos = inscritos_df['documento'] == '2465329551202'
mask_documentos_long = base_longitudinal['documento'] == '2465329551202'

In [ ]:
inscritos_df[mask_documentos]

,documento,nombre,correo,programa,periodo_ingreso,ciclo,estado_ciclo,fase_actual
13,2465329551202,"orozco sanchez de gomez, alba nohemi",aorozco@unis.edu.gt,manes,9243,9243,cn,qa o posterior
11,2465329551202,"orozco sanchez de gomez, alba nohemi",aorozco@unis.edu.gt,manes,9243,9251,cn,qa o posterior
12,2465329551202,"orozco sanchez de gomez, alba nohemi",aorozco@unis.edu.gt,manes,9243,9252,cn,qa o posterior
14,2465329551202,"orozco sanchez de gomez, alba nohemi",aorozco@unis.edu.gt,manes,9243,9253,cn,qa o posterior
